<a href="https://colab.research.google.com/github/Ramyan-Iz/KAA/blob/main/%D0%91%D0%B5%D0%B7%D0%BE%D0%BF%D0%B0%D1%81%D0%BD%D0%BE%D0%B5_%D0%BE%D0%B1%D1%80%D0%B0%D1%89%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B2_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import httpx
import hmac
import hashlib
import urllib.parse
from datetime import datetime
from typing import Optional, List, Dict, Any
from pydantic import BaseModel, Field


# Pydantic модели согласно спецификации API
class StudentRecord(BaseModel):
    """Модель записи о студенте из API"""
    CHANGE_DATE: int = Field(..., description="Время внесения изменения в таблицу STUD (мс)")
    ID_STUD: int = Field(..., description="Код студента")
    FIO: str = Field(..., description="ФИО студента")
    DATE_BIRTH: str = Field(..., description="Дата рождения YYYY-MM-DD")
    EDUC_LEVEL: str = Field(..., description="Уровень образования")
    FORM_NAME: str = Field(..., description="Форма обучения")
    FACULTY_NAME: str = Field(..., description="Название факультета/института")
    NUM_BOOK: Optional[str] = Field(None, description="Номер студенческого билета")
    STATUS: int = Field(..., description="0-действующий, 1-отчисленный, 2-выпускник")


class APIResponse(BaseModel):
    """Модель ответа API"""
    status: str
    data: List[StudentRecord]


class HerzenAPI:
    """
    Минималистичный клиент для API Герцена
    Только GET запросы, HMAC-SHA256 подпись, Pydantic валидация
    """

    def __init__(self, client_id: str, secret_key: str):
        """
        Инициализация клиента

        Args:
            client_id: CLIENT_ID из переменных окружения
            secret_key: SECRET_KEY из переменных окружения
        """
        self.client_id = client_id
        self.secret_key = secret_key
        self.base_url = "https://api.herzen.spb.ru"

    def _generate_signature(self, method: str, path: str, params: Dict[str, Any]) -> str:
        """
        Генерация HMAC-SHA256 подписи согласно спецификации

        Args:
            method: HTTP метод (только GET)
            path: Путь запроса
            params: Query параметры

        Returns:
            HMAC-SHA256 подпись в hex формате
        """
        # 1. Сортируем параметры в алфавитном порядке
        sorted_params = sorted(params.items())

        # 2. Формируем строку параметров
        if sorted_params:
            encoded_params = []
            for key, value in sorted_params:
                encoded_key = urllib.parse.quote(str(key), safe='')
                encoded_value = urllib.parse.quote(str(value), safe='')
                encoded_params.append(f"{encoded_key}={encoded_value}")

            query_string = '?' + '&'.join(encoded_params)
        else:
            query_string = ''

        # 3. Формируем строку для подписи: <метод><путь URI>[?параметры]
        # Тело запроса не нужно, так как только GET
        string_to_sign = f"{method.upper()}{path}{query_string}"

        # 4. Генерируем HMAC-SHA256 подпись
        signature = hmac.new(
            self.secret_key.encode('utf-8'),
            string_to_sign.encode('utf-8'),
            hashlib.sha256
        ).hexdigest()

        return signature

    def get_changes(
        self,
        start_date: int,
        end_date: Optional[int] = None,
        timeout: int = 30
    ) -> APIResponse:
        """
        Получение изменений студентов за период

        Args:
            start_date: Начало периода (UNIX timestamp в миллисекундах, обязательный)
            end_date: Конец периода (UNIX timestamp в миллисекундах, необязательный)
            timeout: Таймаут запроса в секундах

        Returns:
            APIResponse с данными студентов

        Notes:
            Используется полуоткрытый интервал: [start_date, end_date)
        """
        # Формируем параметры запроса
        params = {'start_date': start_date}
        if end_date is not None:
            params['end_date'] = end_date

        # Формируем полный URL
        path = "/api/ldap/changes"
        url = f"{self.base_url}{path}"

        # Генерируем подпись
        signature = self._generate_signature("GET", path, params)

        # Формируем заголовки
        headers = {
            'X-Client-ID': self.client_id,
            'X-Signature': signature,
            'Accept': 'application/json',
        }

        # Выполняем GET запрос
        with httpx.Client(timeout=timeout) as client:
            response = client.get(url, params=params, headers=headers)
            response.raise_for_status()

            # Парсим JSON и валидируем через Pydantic
            response_data = response.json()

            # Проверяем статус ответа
            if response_data.get('status') != 'Success':
                error_msg = response_data.get('message', 'Неизвестная ошибка')
                raise Exception(f"API Error: {error_msg}")

            # Валидируем через Pydantic модели
            return APIResponse(**response_data)


# Асинхронная версия (если нужна)
class HerzenAPIAsync:
    """Асинхронная версия клиента"""

    def __init__(self, client_id: str, secret_key: str):
        self.client_id = client_id
        self.secret_key = secret_key
        self.base_url = "https://api.herzen.spb.ru"

    def _generate_signature(self, method: str, path: str, params: Dict[str, Any]) -> str:
        """Тот же метод генерации подписи"""
        sorted_params = sorted(params.items())

        if sorted_params:
            encoded_params = []
            for key, value in sorted_params:
                encoded_key = urllib.parse.quote(str(key), safe='')
                encoded_value = urllib.parse.quote(str(value), safe='')
                encoded_params.append(f"{encoded_key}={encoded_value}")

            query_string = '?' + '&'.join(encoded_params)
        else:
            query_string = ''

        string_to_sign = f"{method.upper()}{path}{query_string}"

        signature = hmac.new(
            self.secret_key.encode('utf-8'),
            string_to_sign.encode('utf-8'),
            hashlib.sha256
        ).hexdigest()

        return signature

    async def get_changes(
        self,
        start_date: int,
        end_date: Optional[int] = None,
        timeout: int = 30
    ) -> APIResponse:
        """Асинхронное получение изменений"""
        params = {'start_date': start_date}
        if end_date is not None:
            params['end_date'] = end_date

        path = "/api/ldap/changes"
        url = f"{self.base_url}{path}"

        signature = self._generate_signature("GET", path, params)

        headers = {
            'X-Client-ID': self.client_id,
            'X-Signature': signature,
            'Accept': 'application/json',
        }

        async with httpx.AsyncClient(timeout=timeout) as client:
            response = await client.get(url, params=params, headers=headers)
            response.raise_for_status()

            response_data = response.json()

            if response_data.get('status') != 'Success':
                error_msg = response_data.get('message', 'Неизвестная ошибка')
                raise Exception(f"API Error: {error_msg}")

            return APIResponse(**response_data)


# Пример использования
def example_usage():
    """Пример использования клиента"""

    # Конфигурация (заменить на реальные значения!)
    CLIENT_ID = "ваш_client_id"  # ← ЗАМЕНИТЬ
    SECRET_KEY = "ваш_secret_key"  # ← ЗАМЕНИТЬ

    # Создаем клиент
    api = HerzenAPI(CLIENT_ID, SECRET_KEY)

    # Пример 1: Получить изменения за последние 24 часа
    end_time = int(datetime.now().timestamp() * 1000)
    start_time = int((datetime.now().timestamp() - 24 * 3600) * 1000)

    try:
        response = api.get_changes(start_time, end_time)
        print(f"Статус: {response.status}")
        print(f"Получено записей: {len(response.data)}")

        # Работаем с данными через Pydantic
        for record in response.data[:3]:  # Показать первые 3 записи
            print(f"\nСтудент: {record.FIO}")
            print(f"Факультет: {record.FACULTY_NAME}")
            print(f"Статус: {record.STATUS}")

    except httpx.HTTPStatusError as e:
        print(f"HTTP ошибка: {e.response.status_code}")
    except Exception as e:
        print(f"Ошибка: {e}")


# Пример асинхронного использования
async def async_example():
    """Пример асинхронного использования"""

    CLIENT_ID = "ваш_client_id"  # ← ЗАМЕНИТЬ
    SECRET_KEY = "ваш_secret_key"  # ← ЗАМЕНИТЬ

    api = HerzenAPIAsync(CLIENT_ID, SECRET_KEY)

    # Получить все изменения с начала 2024 года
    start_date = int(datetime(2024, 1, 1).timestamp() * 1000)

    try:
        response = await api.get_changes(start_date)
        print(f"Записей с 2024-01-01: {len(response.data)}")

    except Exception as e:
        print(f"Ошибка: {e}")


if __name__ == "__main__":
    # Для тестирования (нужно указать реальные ключи)
    print("Для работы укажите реальные CLIENT_ID и SECRET_KEY")
    # example_usage()

Для работы укажите реальные CLIENT_ID и SECRET_KEY
